<a href="https://colab.research.google.com/github/Kanika-0905/kanika-codeboosters-2026/blob/main/dataengineering_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark --quiet
print("PySpark installation completed!")

PySpark installation completed!


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month,to_date,col,round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

spark=SparkSession.builder \
      .appName('Day4_BigData_Sales') \
      .config('spark.sql.adaaptive.enabled','true') \
      .getOrCreate()
print(f'Spark version : {spark.version}')
print(f'SparkSession : ACTIVE')
print(f'Application  : {spark.sparkContext.appName}')

Spark version : 4.0.2
SparkSession : ACTIVE
Application  : Day4_BigData_Sales


In [ ]:
df_bronze = spark.read \
     .option('header','true') \
     .option('inferSchema','true') \
     .csv('large_sales_data.csv')
print('===BRONZE LAYER - raw data ===')
print(f'Rows :{df_bronze.count()}')

print(f'Columns : {len(df_bronze.columns)}')
print(f'Columns : {df_bronze.columns}')
print()
df_bronze.printSchema()

===BRONZE LAYER - raw data ===
Rows :5000
Columns : 13
Columns : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
print('First 5 rows:')
df_bronze.show(5,truncate=False)
print('\n')
print(df_bronze.tail(5))

First 5 rows:
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Ku

In [ ]:
print('Basic statistics for numeric columns:')
df_bronze.select('quantity','unit_price','revenue').describe().show()

df_bronze = df_bronze.withColumn('total_sales', df_bronze['unit_price'] + df_bronze['revenue'])
df_bronze.select('unit_price','revenue','total_sales').show()

Basic statistics for numeric columns:
+-------+-----------------+------------------+------------------+
|summary|         quantity|        unit_price|           revenue|
+-------+-----------------+------------------+------------------+
|  count|             5000|              5000|              5000|
|   mean|           7.9536|          12496.86|          99169.52|
| stddev|4.275313169878912|14857.384309295603|145972.97195261103|
|    min|                1|               600|               600|
|    max|               15|             45000|            675000|
+-------+-----------------+------------------+------------------+

+----------+-------+-----------+
|unit_price|revenue|total_sales|
+----------+-------+-----------+
|     22000| 264000|     286000|
|     12000| 120000|     132000|
|       800|   8000|       8800|
|     32000| 160000|     192000|
|      3500|  14000|      17500|
|      2500|  25000|      27500|
|       600|   5400|       6000|
|     45000| 585000|     630000|
|   

In [ ]:
df_bronze.write \
       .mode('overwrite') \
       .parquet('sales_bronze.parquet')
print('Bronze Parquet saved: sales_bronze.parquet')

import os

def get_dir_size(path):
  if os.path.isfile(path):
    return os.path.getsize(path) / 1024
  total = 0
  for dirpath,dirnames,filenames in os.walk(path):
    for f in filenames:
      total += os.path.getsize(os.path.join(dirpath,f))
  return total/1024
csv_size =get_dir_size('large_sales_data.csv')
parquet_size = get_dir_size('sales_bronze.parquet')
reduction = (1- parquet_size / csv_size)*100

print(f'Csv size : {csv_size:.1f} KB')
print(f'Parquet size : {parquet_size:.1f} KB')
print(f'Reduction : {reduction:.1f}% smaller')
print(f'\nAt 1TB scale: CSV=1000 GB -> Parquet={1000*(1-reduction/100):.0f} GB')

Bronze Parquet saved: sales_bronze.parquet
Csv size : 529.3 KB
Parquet size : 60.2 KB
Reduction : 88.6% smaller

At 1TB scale: CSV=1000 GB -> Parquet=114 GB


In [ ]:
df_bronze.filter(df_bronze.quantity > 5).show()
df_bronze.where(df_bronze.revenue > 50000).show()
df_bronze.filter(F.col("quantity") > 5).show()

+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+-----------+
|order_id|customer_name|   product|   category|quantity|unit_price|revenue|order_date|     city|region|   sales_rep|  payment_method|order_status|total_sales|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+-----------+
|    1001|  Sneha Reddy|   Monitor|Electronics|      12|     22000| 264000|2023-05-21|   Mumbai|  West| Meera Patel|             UPI|   Delivered|     286000|
|    1002| Ramesh Kumar|   Printer|Electronics|      10|     12000| 120000|2023-08-05|    Delhi| North| Anil Sharma|     Credit Card|     Shipped|     132000|
|    1003| Rahul Mishra|     Mouse|Accessories|      10|       800|   8000|2023-01-14|Ahmedabad|  West| Meera Patel|Cash on Delivery|     Shipped|       8800|
|    1006|   Suresh Rao|    Webcam|Accessories

In [ ]:
df_bronze.groupBy("category") \
    .agg(F.sum("revenue").alias("total_rev"),
     F.count("*").alias("orders"),
     F.avg("revenue").alias("avg_revenue")) \
     .orderBy("total_rev", ascending=False)

DataFrame[category: string, total_rev: bigint, orders: bigint, avg_revenue: double]

In [ ]:
df_silver = df_bronze \
.dropDuplicates() \
.dropna(subset=['order_id','product','revenue'])

df_silver=df_silver.withColumn('order_date',to_date(col('order_date'),'dd-MM-yyyy')
)
df_silver=df_silver \
.withColumn('order_year',year(col('order_date'))) \
.withColumn('order_month',month(col('order_date')))

df_silver = df_silver.withColumn( 'revenue_category',F.when(col('revenue')>40000,'High')
.when(col('revenue')>10000,'Medium')
.otherwise('Low'))

print(f'Silver layer rows: {df_silver.count()}')
print('New columns added: order_year, order_month', 'revenue_category')
df_silver.select('product', 'revenue', 'order_year', 'order_month', 'revenue_category').show(8)

Silver layer rows: 5000
New columns added: order_year, order_month revenue_category
+-------+-------+----------+-----------+----------------+
|product|revenue|order_year|order_month|revenue_category|
+-------+-------+----------+-----------+----------------+
|Monitor| 242000|      2023|         11|            High|
| Webcam|  27500|      2023|          1|          Medium|
|Printer| 156000|      2023|          9|            High|
|Printer|  84000|      2023|          8|            High|
|Monitor| 110000|      2023|         11|            High|
|Printer| 132000|      2023|          6|            High|
|Printer| 168000|      2023|          4|            High|
|  Mouse|   9600|      2023|          2|             Low|
+-------+-------+----------+-----------+----------------+
only showing top 8 rows


In [ ]:
df_silver.write \
.mode('overwrite') \
.parquet('sales_silver.parquet')
print('Silver Parquet saved: sales_silver.parquet')
print(f'Silver size : {get_dir_size("sales_silver.parquet"):.1f}KB')
print(f'bronze size : {get_dir_size("sales_bronze.parquet"):.1f}KB')
df_verify = spark.read.parquet('sales_silver.parquet')
print(f'Read-back rows: {df_verify.count()} (Should match  silver count)')
df_verify.printSchema()

Silver Parquet saved: sales_silver.parquet
Silver size : 64.9KB
bronze size : 60.2KB
Read-back rows: 5000 (Should match  silver count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- total_sales: integer (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)



In [ ]:
top_products = df_silver \
.groupBy('product') \
.agg(F.sum('revenue').alias('total_revenue'),
     F.count('order_id').alias('num_orders'),
     spark_round(F.avg('revenue'),2).alias('avg_order_revenue')) \
     .orderBy('total_revenue') \
     .limit(10)

print('Top 5 Products by revenue')
top_products.show(truncate=False)


Top 5 Products by revenue
+----------+-------------+----------+-----------------+
|product   |total_revenue|num_orders|avg_order_revenue|
+----------+-------------+----------+-----------------+
|USB Hub   |2447400      |527       |4644.02          |
|Mouse     |3207200      |492       |6518.7           |
|Keyboard  |4878000      |495       |9854.55          |
|Webcam    |10982500     |532       |20643.8          |
|Headphones|13541500     |481       |28152.81         |
|Speaker   |16317000     |470       |34717.02         |
|Printer   |44544000     |488       |91278.69         |
|Monitor   |82126000     |481       |170740.12        |
|Tablet    |135104000    |532       |253954.89        |
|Laptop    |182700000    |502       |363944.22        |
+----------+-------------+----------+-----------------+



In [ ]:
top_regions=df_silver \
.groupBy('region') \
.agg(F.sum('revenue').alias("total_revenue"),
   F.count('order_id').alias("total_orders"),
     F.countDistinct('customer_name')).alias("unique_customers") \
.orderBy('total_revenue',ascending=False)

top_regions.show(truncate=False)


+------+-------------+------------+-----------------------------+
|region|total_revenue|total_orders|count(DISTINCT customer_name)|
+------+-------------+------------+-----------------------------+
|West  |198275600    |2021        |15                           |
|South |147145900    |1483        |15                           |
|North |99878400     |995         |15                           |
|East  |50547700     |501         |15                           |
+------+-------------+------------+-----------------------------+



In [ ]:
order_month=df_silver \
.groupBy(
    F.month('order_date'),
         ) \
.agg(F.sum('revenue').alias("monthly_revenue"),
   F.count('order_id').alias("monthly_orders")) \
.orderBy(month('order_date'))

order_month.show(truncate=False)

+-----------------+---------------+--------------+
|month(order_date)|monthly_revenue|monthly_orders|
+-----------------+---------------+--------------+
|1                |41068200       |423           |
|2                |34485400       |375           |
|3                |40031200       |451           |
|4                |38857100       |390           |
|5                |39984500       |423           |
|6                |40707400       |390           |
|7                |42640700       |405           |
|8                |43718500       |418           |
|9                |37640200       |398           |
|10               |47839000       |479           |
|11               |44577100       |419           |
|12               |44298300       |429           |
+-----------------+---------------+--------------+



In [ ]:
order_month = df_silver \
    .groupBy(
        F.month('order_date').alias('month_num'),
        F.date_format('order_date', 'MMMM').alias('order_month')
    ) \
    .agg(
        F.sum('revenue').alias('monthly_revenue'),
        F.count('order_id').alias('monthly_orders')
    ) \
    .orderBy('month_num') \
    .select('order_month', 'monthly_revenue', 'monthly_orders')

order_month.show(truncate=False)

+-----------+---------------+--------------+
|order_month|monthly_revenue|monthly_orders|
+-----------+---------------+--------------+
|January    |41068200       |423           |
|February   |34485400       |375           |
|March      |40031200       |451           |
|April      |38857100       |390           |
|May        |39984500       |423           |
|June       |40707400       |390           |
|July       |42640700       |405           |
|August     |43718500       |418           |
|September  |37640200       |398           |
|October    |47839000       |479           |
|November   |44577100       |419           |
|December   |44298300       |429           |
+-----------+---------------+--------------+

